# Azure 嵌入示例

> 注意：openai 库已有更新的版本。请参阅 https://github.com/openai/openai-python/discussions/742

本示例将介绍如何使用 Azure OpenAI 服务生成嵌入。

## 设置

首先，安装所需的依赖。

! pip install "openai>=0.28.1,<1.0.0"

为了让后续部分正常运行，我们需要先完成一些设置。先从 `api_base` 和 `api_version` 开始。要找到你的 `api_base`，请访问 https://portal.azure.com, 找到你的资源，然后在 "Resource Management" -> "Keys and Endpoints" 下查找 "Endpoint" 的值。

In [ ]:
import os
import openai

In [ ]:
openai.api_version = '2023-05-15'
openai.api_base = '' # Please add your endpoint here

接下来需要设置 `api_type` 和 `api_key`。我们可以从门户获取密钥，也可以通过 Microsoft Active Directory 身份验证获取。根据所用方式，`api_type` 为 `azure` 或 `azure_ad`。

### 设置：门户
先来看如何从门户获取密钥。访问 https://portal.azure.com, 找到你的资源，然后在 "Resource Management" -> "Keys and Endpoints" 下查找其中一个 "Keys" 的值。

In [ ]:
openai.api_type = 'azure'
openai.api_key = os.environ["OPENAI_API_KEY"]

> 注意：在本示例中，我们通过在代码中设置变量来让库使用 Azure API。在开发时，可以考虑改为设置以下环境变量：

```
OPENAI_API_BASE
OPENAI_API_KEY
OPENAI_API_TYPE
OPENAI_API_VERSION
```

### （可选）设置：Microsoft Active Directory 身份验证
现在来看如何通过 Microsoft Active Directory 身份验证获取密钥。如果你想使用 Active Directory 身份验证而不是门户中的密钥，请取消注释下面的代码。

In [ ]:
# from azure.identity import DefaultAzureCredential

# default_credential = DefaultAzureCredential()
# token = default_credential.get_token("https://cognitiveservices.azure.com/.default")

# openai.api_type = 'azure_ad'
# openai.api_key = token.token

令牌在一段时间内有效，过期后即失效。为了确保每个请求都携带有效的令牌，你可以通过挂接 requests.auth 来刷新即将过期的令牌：

In [ ]:
import typing
import time
import requests
if typing.TYPE_CHECKING:
    from azure.core.credentials import TokenCredential

class TokenRefresh(requests.auth.AuthBase):

    def __init__(self, credential: "TokenCredential", scopes: typing.List[str]) -> None:
        self.credential = credential
        self.scopes = scopes
        self.cached_token: typing.Optional[str] = None

    def __call__(self, req):
        if not self.cached_token or self.cached_token.expires_on - time.time() < 300:
            self.cached_token = self.credential.get_token(*self.scopes)
        req.headers["Authorization"] = f"Bearer {self.cached_token.token}"
        return req

session = requests.Session()
session.auth = TokenRefresh(default_credential, ["https://cognitiveservices.azure.com/.default"])

openai.requestssession = session

## 部署
在本节中，我们将创建一个可用于生成嵌入的部署。

### 部署：手动创建
我们使用 `text-similarity-curie-001` 模型创建一个部署。在门户中进入你的资源，在 "Resource Management" -> "Model deployments" 下创建新部署。

In [ ]:
deployment_id = '' # Fill in the deployment id from the portal here

### 部署：列出
由于创建新部署需要较长时间，我们在订阅中查找一个已经成功完成的部署。

In [ ]:
print('While deployment running, selecting a completed one that supports embeddings.')
deployment_id = None
result = openai.Deployment.list()
for deployment in result.data:
    if deployment["status"] != "succeeded":
        continue
    
    model = openai.Model.retrieve(deployment["model"])
    if model["capabilities"]["embeddings"] != True:
        continue
    
    deployment_id = deployment["id"]
    break

if not deployment_id:
    print('No deployment with status: succeeded found.')
else:
    print(f'Found a succeeded deployment that supports embeddings with id: {deployment_id}.')

### 嵌入
现在向该部署发送一个嵌入示例请求。

In [ ]:
embeddings = openai.Embedding.create(deployment_id=deployment_id,
                                     input="The food was delicious and the waiter...")
                                
print(embeddings)